# 09. Advanced attention — sparse compute and a Kimi-K3 miniature attention stack

This notebook now implements the structures rather than naming them:

- local and routed sparse attention
- Kimi Delta Attention (KDA) recurrent state with channel-wise decay
- short causal convolution and output gate
- Gated MLA with NoPE content path and output gate
- exact miniature **3 KDA : 1 Gated MLA** layer pattern
- Block Attention Residuals across depth


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. Sparse compute without dense-then-mask


In [ ]:
def routed_block_attention(q, k, v, block_size=4, selected_blocks=2):
    sequence, dim = q.shape
    blocks = sequence // block_size

    q_blocks = q.view(blocks, block_size, dim)
    k_blocks = k.view(blocks, block_size, dim)
    v_blocks = v.view(blocks, block_size, dim)

    q_summary = q_blocks.mean(dim=1)
    k_summary = k_blocks.mean(dim=1)
    coarse = q_summary @ k_summary.T / math.sqrt(dim)
    chosen = coarse.topk(selected_blocks, dim=-1).indices

    outputs = []
    score_count = 0

    for query_block_id in range(blocks):
        key_ids = chosen[query_block_id]
        selected_k = k_blocks[key_ids].reshape(-1, dim)
        selected_v = v_blocks[key_ids].reshape(-1, dim)
        query_block = q_blocks[query_block_id]

        scores = query_block @ selected_k.T / math.sqrt(dim)
        weights = scores.softmax(dim=-1)
        outputs.append(weights @ selected_v)
        score_count += scores.numel()

    return torch.cat(outputs, dim=0), chosen, score_count


q = torch.randn(16, 8, device=device)
k = torch.randn_like(q)
v = torch.randn_like(q)
sparse_out, chosen, count = routed_block_attention(q, k, v)
print("chosen blocks:", chosen)
print("fine scores:", count, "vs dense:", 16 * 16)


## 2. KDA recurrence

For each head, KDA keeps a fixed-size matrix state.
The state is first forgotten by a **channel-wise decay vector** `alpha_t`,
then corrected by the delta rule.

`S_t = (I - beta_t k_t k_t^T) Diag(alpha_t) S_(t-1) + beta_t k_t v_t^T`

The code uses the row-vector-equivalent ordering so the dimensions remain explicit.


In [ ]:
def kda_scan(q, k, v, alpha, beta):
    batch, heads, length, key_dim = q.shape
    value_dim = v.size(-1)

    state = torch.zeros(
        batch,
        heads,
        key_dim,
        value_dim,
        device=q.device,
        dtype=q.dtype,
    )
    outputs = []

    for time_index in range(length):
        q_t = F.normalize(q[:, :, time_index], dim=-1)
        k_t = F.normalize(k[:, :, time_index], dim=-1)
        v_t = v[:, :, time_index]

        alpha_t = alpha[:, :, time_index]
        beta_t = beta[:, :, time_index]

        decayed_state = alpha_t[..., None] * state

        predicted = torch.einsum(
            "bhkv,bhk->bhv",
            decayed_state,
            k_t,
        )
        error = v_t - predicted

        state = (
            decayed_state
            + beta_t[..., None, None]
            * torch.einsum("bhk,bhv->bhkv", k_t, error)
        )
        output_t = torch.einsum(
            "bhkv,bhk->bhv",
            state,
            q_t,
        )
        outputs.append(output_t)

    return torch.stack(outputs, dim=2), state


## 3. Complete small KDA layer

The hidden width and number of heads are reduced, but the important paths remain:

`input projection -> causal depthwise short convolution -> Q/K/V`
`-> full-rank channel-wise decay alpha + beta -> KDA scan -> output gate -> projection`

The lower bound `-5` follows the released K3 configuration.


In [ ]:
class TinyKDA(nn.Module):
    def __init__(self, model_dim=32, heads=4, head_dim=8, short_kernel=4):
        super().__init__()
        self.heads = heads
        self.head_dim = head_dim
        inner = heads * head_dim
        self.short_kernel = short_kernel

        self.norm = nn.RMSNorm(model_dim)
        self.in_proj = nn.Linear(model_dim, inner, bias=False)

        self.short_conv = nn.Conv1d(
            inner,
            inner,
            kernel_size=short_kernel,
            groups=inner,
            bias=True,
        )

        self.q_proj = nn.Linear(inner, inner, bias=False)
        self.k_proj = nn.Linear(inner, inner, bias=False)
        self.v_proj = nn.Linear(inner, inner, bias=False)

        self.alpha_proj = nn.Linear(inner, inner, bias=True)
        self.beta_proj = nn.Linear(inner, heads, bias=True)
        self.output_gate = nn.Linear(inner, inner, bias=True)
        self.out_proj = nn.Linear(inner, model_dim, bias=False)

        self.gate_lower_bound = -5.0

    def forward(self, hidden):
        batch, length, _ = hidden.shape
        inner_hidden = self.in_proj(self.norm(hidden))

        convolution_input = inner_hidden.transpose(1, 2)
        convolution_input = F.pad(
            convolution_input,
            (self.short_kernel - 1, 0),
        )
        convolved = self.short_conv(convolution_input).transpose(1, 2)
        convolved = F.silu(convolved)

        def split_heads(x):
            return x.view(
                batch,
                length,
                self.heads,
                self.head_dim,
            ).transpose(1, 2)

        q = split_heads(self.q_proj(convolved))
        k = split_heads(self.k_proj(convolved))
        v = split_heads(self.v_proj(convolved))

        alpha_logits = self.alpha_proj(convolved).view(
            batch,
            length,
            self.heads,
            self.head_dim,
        ).transpose(1, 2)
        alpha_logits = alpha_logits.clamp_min(self.gate_lower_bound)
        alpha = torch.sigmoid(alpha_logits)

        beta = torch.sigmoid(
            self.beta_proj(convolved)
        ).transpose(1, 2)

        scanned, final_state = kda_scan(q, k, v, alpha, beta)
        scanned = scanned.transpose(1, 2).contiguous().flatten(2)

        gate = torch.sigmoid(self.output_gate(convolved))
        output = self.out_proj(gate * scanned)

        diagnostics = {
            "alpha": alpha,
            "beta": beta,
            "state": final_state,
        }
        return output, diagnostics


kda = TinyKDA().to(device)
hidden = torch.randn(2, 12, 32, device=device)
kda_output, kda_diag = kda(hidden)
print("KDA:", kda_output.shape)
print("alpha:", kda_diag["alpha"].shape)
print("state:", kda_diag["state"].shape)


## 4. Gated MLA with NoPE content attention

K3's released configuration uses MLA without RoPE for this path and adds
an input-dependent output gate. Low-rank Q/KV projections are retained.


In [ ]:
class TinyGatedMLA(nn.Module):
    def __init__(
        self,
        model_dim=32,
        heads=4,
        head_dim=8,
        q_rank=12,
        kv_rank=8,
    ):
        super().__init__()
        self.heads = heads
        self.head_dim = head_dim

        self.norm = nn.RMSNorm(model_dim)
        self.q_down = nn.Linear(model_dim, q_rank, bias=False)
        self.q_up = nn.Linear(q_rank, heads * head_dim, bias=False)

        self.kv_down = nn.Linear(model_dim, kv_rank, bias=False)
        self.kv_up = nn.Linear(
            kv_rank,
            2 * heads * head_dim,
            bias=False,
        )

        self.output_gate = nn.Linear(model_dim, heads * head_dim)
        self.out = nn.Linear(heads * head_dim, model_dim, bias=False)

    def forward(self, hidden):
        batch, length, _ = hidden.shape
        normalized = self.norm(hidden)

        q = self.q_up(self.q_down(normalized)).view(
            batch,
            length,
            self.heads,
            self.head_dim,
        ).transpose(1, 2)

        kv = self.kv_up(self.kv_down(normalized))
        k, v = kv.chunk(2, dim=-1)

        k = k.view(
            batch,
            length,
            self.heads,
            self.head_dim,
        ).transpose(1, 2)
        v = v.view(
            batch,
            length,
            self.heads,
            self.head_dim,
        ).transpose(1, 2)

        attended = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
        )
        attended = attended.transpose(1, 2).contiguous().flatten(2)

        gate = torch.sigmoid(self.output_gate(normalized))
        return self.out(gate * attended)


## 5. Block Attention Residuals integrated into depth


In [ ]:
class BlockDepthAttention(nn.Module):
    def __init__(self, model_dim=32):
        super().__init__()
        self.norm = nn.RMSNorm(model_dim)
        self.query = nn.Parameter(torch.randn(model_dim) * 0.02)

    def forward(self, sources):
        stacked = torch.stack(sources, dim=2)
        normalized = self.norm(stacked)

        scores = torch.einsum(
            "d,btkd->btk",
            self.query,
            normalized,
        ) / math.sqrt(normalized.size(-1))

        weights = scores.softmax(dim=-1)
        retrieved = torch.einsum(
            "btk,btkd->btd",
            weights,
            stacked,
        )
        return retrieved, weights


## 6. K3 miniature: exactly 3 KDA layers + 1 Gated MLA

The full model is much deeper. Here the only reduction is depth/width.
A two-layer depth block is used for the executable Block AttnRes demonstration.


In [ ]:
class K3AttentionLayer(nn.Module):
    def __init__(self, kind, model_dim=32):
        super().__init__()
        self.kind = kind
        self.norm = nn.RMSNorm(model_dim)

        if kind == "kda":
            self.attention = TinyKDA(model_dim=model_dim)
        elif kind == "mla":
            self.attention = TinyGatedMLA(model_dim=model_dim)
        else:
            raise ValueError(kind)

        self.ffn = nn.Sequential(
            nn.RMSNorm(model_dim),
            nn.Linear(model_dim, 4 * model_dim),
            nn.SiLU(),
            nn.Linear(4 * model_dim, model_dim),
        )

    def forward(self, hidden):
        if self.kind == "kda":
            attention_output, _ = self.attention(self.norm(hidden))
        else:
            attention_output = self.attention(self.norm(hidden))

        hidden = hidden + attention_output
        hidden = hidden + self.ffn(hidden)
        return hidden


class TinyK3AttentionStack(nn.Module):
    def __init__(self, model_dim=32, block_size=2):
        super().__init__()
        self.block_size = block_size
        self.pattern = ["kda", "kda", "kda", "mla"]
        self.layers = nn.ModuleList(
            [
                K3AttentionLayer(kind, model_dim)
                for kind in self.pattern
            ]
        )
        self.depth_attention = BlockDepthAttention(model_dim)

    def forward(self, hidden):
        block_sources = [hidden]
        depth_weights = []

        for layer_index, layer in enumerate(self.layers):
            if layer_index > 0 and layer_index % self.block_size == 0:
                hidden, weights = self.depth_attention(block_sources)
                depth_weights.append(weights)

            hidden = layer(hidden)

            if (layer_index + 1) % self.block_size == 0:
                block_sources.append(hidden)

        return hidden, depth_weights


k3_stack = TinyK3AttentionStack().to(device)
k3_output, depth_weights = k3_stack(hidden)

loss = k3_output.square().mean()
loss.backward()

print("pattern:", k3_stack.pattern)
print("output:", k3_output.shape)
print("depth-attention sites:", len(depth_weights))
print("KDA alpha grad:",
      k3_stack.layers[0].attention.alpha_proj.weight.grad.norm().item())
print("MLA output-gate grad:",
      k3_stack.layers[-1].attention.output_gate.weight.grad.norm().item())


## References and provenance

- **Kimi Delta Attention**: delta-rule matrix memory with input-dependent
  channel-wise decay and a short-convolution path.
- **Kimi K3**: released architecture mixes KDA and Gated MLA in a **3:1**
  ratio and uses Attention Residuals across depth.
- Tensor width, sequence length, and depth are reduced here; the above
  computation paths are not replaced by plain DeltaNet or generic attention.
